In [1]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
print("공개 Hugging Face 모델을 인증 없이 사용합니다.")

공개 Hugging Face 모델을 인증 없이 사용합니다.


In [2]:
import shutil
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import subprocess
from pathlib import Path

WORKDIR = Path(os.environ.get("SNU_WORKDIR", "/content/snu_ai"))
DRIVE_ROOT = Path(os.environ.get("SNU_DRIVE_ROOT", "/content/drive/MyDrive/snu ai"))
DATA_ROOT = WORKDIR / "snuaichallenge_data"


def find_drive_file(filename):
    direct = DRIVE_ROOT / filename
    if direct.is_file():
        return direct
    matches = list(Path("/content/drive/MyDrive").rglob(filename))
    return matches[0] if matches else None


# The existing full-train bundle is the source of train.csv and 38,140 train images.
if not (DATA_ROOT / "train.csv").is_file() or not (DATA_ROOT / "train").is_dir():
    train_bundle = find_drive_file("snu_colab_full_train_bundle.tar")
    if train_bundle is None:
        raise FileNotFoundError(
            "snu_colab_full_train_bundle.tar was not found in MyDrive."
        )
    WORKDIR.mkdir(parents=True, exist_ok=True)
    print(f"Extracting existing training bundle: {train_bundle}")
    subprocess.run(
        ["tar", "-xf", str(train_bundle), "-C", str(WORKDIR)],
        check=True,
    )
    DRIVE_ROOT = train_bundle.parent

assert (DATA_ROOT / "train.csv").is_file(), DATA_ROOT / "train.csv"
assert (DATA_ROOT / "train").is_dir(), DATA_ROOT / "train"

# Test data is optional for 80:20 holdout training. Restore it only when its bundle exists.
test_bundle = find_drive_file("snu_colab_test_bundle.tar")
if test_bundle is not None and (
    not (DATA_ROOT / "test.csv").is_file()
    or not (DATA_ROOT / "test").is_dir()
    or not (DATA_ROOT / "sample_submission.csv").is_file()
):
    print(f"Extracting existing test bundle: {test_bundle}")
    subprocess.run(
        ["tar", "-xf", str(test_bundle), "-C", str(WORKDIR)],
        check=True,
    )

DATA_DIR = str(DATA_ROOT)
OUTPUT_DIR = str(DRIVE_ROOT / "snu_qwen3vl_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Training data: {DATA_DIR}")
print(f"Persistent output: {OUTPUT_DIR}")

Extracting existing training bundle: /content/drive/MyDrive/snu ai/snu_colab_full_train_bundle.tar
Extracting existing test bundle: /content/drive/MyDrive/snu ai/snu_colab_test_bundle.tar
Training data: /content/snu_ai/snuaichallenge_data
Persistent output: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs


In [5]:
!pip install -q -U "transformers>=4.57.0" accelerate bitsandbytes "qwen-vl-utils==0.0.14" "scikit-learn==1.8.0" "networkx==3.6.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 178.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 283.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 56.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 106.4 MB/s eta 0:00:0000:0100:01


In [6]:
import os, re, ast, random, hashlib, time
import numpy as np
import pandas as pd
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict
from tqdm.auto import tqdm
import networkx as nx
from sklearn.model_selection import StratifiedGroupKFold

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForImageTextToText, AutoProcessor
from qwen_vl_utils import process_vision_info

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"
MODEL_REVISION = "0c351dd01ed87e9c1b53cbc748cba10e6187ff3b"
IMAGE_PATCH_SIZE = 16
MIN_PIXELS = 4 * 32 * 32
MAX_PIXELS = 256 * 32 * 32
VALIDATION_RATIO = 0.2
MAX_SAMPLES_PER_HASH = 3

train = pd.read_csv(f'{DATA_DIR}/train.csv')
train['answer_tuple'] = train['Answer'].apply(lambda x: str(ast.literal_eval(x)))

In [7]:
def compute_md5(filepath):
    with open(filepath, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

def get_all_image_records(df, img_dir):
    records = []
    for _, row in df.iterrows():
        for i in range(1, 5):
            path = f"{img_dir}/{row['Id']}/{row[f'Input_{i}']}"
            records.append({'sample_id': row['Id'], 'path': path})
    return records

print("Train 이미지 해시 계산 중...")
train_records = get_all_image_records(train, f'{DATA_DIR}/train')
paths = [r['path'] for r in train_records]
with ThreadPoolExecutor(max_workers=32) as executor:
    hashes = list(tqdm(executor.map(compute_md5, paths), total=len(paths)))
for r, h in zip(train_records, hashes):
    r['hash'] = h

hash_counter = defaultdict(set)
for r in train_records:
    hash_counter[r['hash']].add(r['sample_id'])

hash_to_samples = {h: s for h, s in hash_counter.items() if len(s) <= MAX_SAMPLES_PER_HASH}
excluded = sum(1 for h, s in hash_counter.items() if len(s) > MAX_SAMPLES_PER_HASH)
print(f"제외된 해시(빈번 재사용) 수: {excluded}")

G = nx.Graph()
G.add_nodes_from(train['Id'].unique())
for h, sample_ids in hash_to_samples.items():
    sample_ids = list(sample_ids)
    for i in range(len(sample_ids)):
        for j in range(i+1, len(sample_ids)):
            G.add_edge(sample_ids[i], sample_ids[j])

components = list(nx.connected_components(G))
print(f"총 {len(components)}개 클러스터, 최대 크기={max(len(c) for c in components)}")

id_to_cluster = {sid: cid for cid, comp in enumerate(components) for sid in comp}
train['cluster_id'] = train['Id'].map(id_to_cluster)

# StratifiedGroupKFold is used only to select one deterministic 20% holdout.
# This is one 80:20 experiment, not five-fold cross-validation.
holdout_partitions = round(1 / VALIDATION_RATIO)
assert holdout_partitions == 5 and abs(VALIDATION_RATIO - 0.2) < 1e-9
splitter = StratifiedGroupKFold(
    n_splits=holdout_partitions, shuffle=True, random_state=SEED
)
train_idx, val_idx = next(
    splitter.split(train, train['answer_tuple'], groups=train['cluster_id'])
)
train['split'] = 'train'
train.loc[val_idx, 'split'] = 'validation'

train_groups = set(train.loc[train_idx, 'cluster_id'])
validation_groups = set(train.loc[val_idx, 'cluster_id'])
assert train_groups.isdisjoint(validation_groups)
assert train.loc[train_idx, 'answer_tuple'].nunique() == 24
assert train.loc[val_idx, 'answer_tuple'].nunique() == 24
print(f"Train: {len(train_idx)}, validation: {len(val_idx)}")
print(f"MD5 clusters crossing the split: {len(train_groups & validation_groups)}")
print("Validation target counts:")
print(train.loc[val_idx, 'answer_tuple'].value_counts().sort_index())
print("Validation No_ordering rows:", int(train.loc[val_idx, 'No_ordering'].sum()))

Train 이미지 해시 계산 중...


  0%|          | 0/38140 [00:00<?, ?it/s]

제외된 해시(빈번 재사용) 수: 2081
총 8147개 클러스터, 최대 크기=14
Train: 7628, validation: 1907
MD5 clusters crossing the split: 0
Validation target counts:
answer_tuple
[1, 2, 3, 4]    296
[1, 2, 4, 3]     71
[1, 3, 2, 4]     72
[1, 3, 4, 2]     69
[1, 4, 2, 3]     69
[1, 4, 3, 2]     71
[2, 1, 3, 4]     68
[2, 1, 4, 3]     73
[2, 3, 1, 4]     61
[2, 3, 4, 1]     73
[2, 4, 1, 3]     70
[2, 4, 3, 1]     69
[3, 1, 2, 4]     72
[3, 1, 4, 2]     67
[3, 2, 1, 4]     72
[3, 2, 4, 1]     68
[3, 4, 1, 2]     77
[3, 4, 2, 1]     70
[4, 1, 2, 3]     63
[4, 1, 3, 2]     67
[4, 2, 1, 3]     71
[4, 2, 3, 1]     71
[4, 3, 1, 2]     74
[4, 3, 2, 1]     73
Name: count, dtype: int64
Validation No_ordering rows: 296


In [8]:
CLAUSE_PATTERN = re.compile(
    r'\b(then|after which|after that|and finally|finally|next|before that|followed by|first|before|after)\b,?',
    flags=re.IGNORECASE
)

def split_clauses(sentence, max_clauses=4):
    sentence = sentence.strip()
    parts = CLAUSE_PATTERN.split(sentence)
    clauses = []
    current = parts[0].strip(' ,;.')
    for i in range(1, len(parts), 2):
        if current:
            clauses.append(current)
        current = parts[i+1].strip(' ,;.')
    if current:
        clauses.append(current)
    clauses = [c for c in clauses if c]
    if len(clauses) > max_clauses:
        clauses = clauses[:max_clauses-1] + [' '.join(clauses[max_clauses-1:])]
    if not clauses:
        clauses = [sentence]
    return clauses

In [9]:
STRUCTURED_HINT = (
    "This caption describes distinct sequential stages, often marked by words like "
    "'then', 'after which', or 'finally'. Match each stage of the story to its corresponding image."
)
CONTINUOUS_HINT = (
    "This caption describes a single continuous action or a brief moment. "
    "Focus on subtle visual cues such as camera movement direction, object position changes, "
    "hand/body motion, and lighting changes to infer the temporal order."
)

def get_prompt_message(row, image_dir):
    img_files = [row['Input_1'], row['Input_2'], row['Input_3'], row['Input_4']]
    sentence = row['Sentence']
    clause_count = len(split_clauses(sentence))
    hint = STRUCTURED_HINT if clause_count >= 3 else CONTINUOUS_HINT

    content = []
    for i, img_file in enumerate(img_files):
        img_path = os.path.join(image_dir, row['Id'], img_file)
        content.append({
            "type": "image",
            "image": img_path,
            "min_pixels": MIN_PIXELS,
            "max_pixels": MAX_PIXELS,
        })
        content.append({"type": "text", "text": f"\nImage {i+1}\n"})

    user_text = (
        f"Thinking about the sentence: \"{sentence}\"\n"
        f"{hint}\n"
        "Look at the 4 images above labeled Image 1 to Image 4. "
        "Determine the correct chronological order of these images to match the sentence. "
        "Provide the answer ONLY as a Python list of integers. "
        "Example: [1, 2, 3, 4]"
    )
    content.append({"type": "text", "text": user_text})
    return [{"role": "user", "content": content}]


def format_training_sample(row, img_dir):
    messages = get_prompt_message(row, img_dir)
    answer = ast.literal_eval(row['Answer'])
    temporal_order = [0, 0, 0, 0]
    for input_idx, pos in enumerate(answer):
        temporal_order[pos - 1] = input_idx + 1
    target_text = str(temporal_order)
    messages.append({"role": "assistant", "content": [{"type": "text", "text": target_text}]})
    return messages, target_text


def parse_model_output(output_text):
    try:
        start_idx = output_text.find('[')
        end_idx = output_text.rfind(']')
        if start_idx != -1 and end_idx != -1:
            result = ast.literal_eval(output_text[start_idx:end_idx+1])
            if isinstance(result, list) and len(result) == 4 and sorted(result) == [1,2,3,4]:
                submission_answer = [0]*4
                for index, image_num in enumerate(result):
                    submission_answer[image_num - 1] = index + 1
                return submission_answer, True
    except:
        pass
    return [1,2,3,4], False

In [10]:
# Full fine-tuning GPU inspection: run this before loading the model.
assert torch.cuda.is_available(), "CUDA GPU가 필요합니다."
gpu = torch.cuda.get_device_properties(0)
GPU_NAME = gpu.name
GPU_VRAM_GB = gpu.total_memory / 1024**3
print(f"GPU: {GPU_NAME}")
print(f"VRAM: {GPU_VRAM_GB:.1f} GiB")
print("8B VLM 전체 학습은 G4 RTX PRO 6000 96GB 또는 A100 80GB급 단일 GPU용입니다.")
assert GPU_VRAM_GB >= 70, (
    f"감지된 VRAM은 {GPU_VRAM_GB:.1f} GiB입니다. "
    "이 설정의 true full fine-tuning에는 70 GiB 이상이 필요합니다."
)

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GiB
8B VLM 전체 학습은 G4 RTX PRO 6000 96GB 또는 A100 80GB급 단일 GPU용입니다.


In [11]:
# 위 GPU 이름과 VRAM을 확인한 뒤에만 True로 바꾸세요.
FULL_FINETUNE_GPU_CONFIRMED = "NVIDIA RTX PRO 6000 Blackwell Server Edition"
assert FULL_FINETUNE_GPU_CONFIRMED, (
    "GPU 확인 전에는 학습 모델을 로드하지 않습니다. "
    "출력된 GPU/VRAM을 확인한 뒤 FULL_FINETUNE_GPU_CONFIRMED=True로 바꾸세요."
)

In [12]:
import shutil

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')
print(f"GPU ?? ??: {GPU_NAME} ({GPU_VRAM_GB:.1f} GiB)")
print('BF16 + SDPA + TF32 fallback + local SSD input enabled')

GPU ?? ??: NVIDIA RTX PRO 6000 Blackwell Server Edition (95.0 GiB)
BF16 + SDPA + TF32 fallback + local SSD input enabled


In [13]:
assert FULL_FINETUNE_GPU_CONFIRMED
processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
)

# device_map='auto'와 양자화 모델은 사용하지 않는다. BF16 원본 가중치를 한 GPU에 올린다.
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
).to(DEVICE)
model.config.use_cache = False

# Vision tower와 language model을 포함해 모든 파라미터를 학습 대상으로 강제한다.
for parameter in model.parameters():
    parameter.requires_grad_(True)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params
adapter_params = [name for name, _ in model.named_parameters() if 'lora_' in name.lower()]
print(f"전체 파라미터: {total_params:,}")
print(f"학습 파라미터: {trainable_params:,} ({trainable_params / total_params:.2%})")
print(f"동결 파라미터: {frozen_params:,}")
assert trainable_params == total_params, "일부 파라미터가 동결되어 있습니다."
assert not adapter_params, "LoRA/PEFT 파라미터가 발견되었습니다."
assert not hasattr(model, 'peft_config'), "PEFT 래퍼가 발견되었습니다."
print("검증 통과: LoRA가 아닌 true full-parameter fine-tuning 모델입니다.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/67.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

전체 파라미터: 8,767,123,696
학습 파라미터: 8,767,123,696 (100.00%)
동결 파라미터: 0
검증 통과: LoRA가 아닌 true full-parameter fine-tuning 모델입니다.


In [14]:
class QwenOrderingDataset(Dataset):
    def __init__(self, df, img_dir, processor):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        messages, target_text = format_training_sample(row, self.img_dir)

        full_text = self.processor.apply_chat_template(messages, tokenize=False)
        prompt_only = self.processor.apply_chat_template(messages[:1], tokenize=False, add_generation_prompt=True)

        image_inputs, _ = process_vision_info(
            messages, image_patch_size=IMAGE_PATCH_SIZE
        )

        full_enc = self.processor(
            text=[full_text], images=image_inputs, return_tensors="pt",
            padding=False, do_resize=False,
        )
        prompt_enc = self.processor(
            text=[prompt_only], images=image_inputs, return_tensors="pt",
            padding=False, do_resize=False,
        )

        input_ids = full_enc['input_ids'][0]
        labels = input_ids.clone()
        prompt_len = prompt_enc['input_ids'].shape[1]
        labels[:prompt_len] = -100

        return {
            'input_ids': input_ids,
            'attention_mask': full_enc['attention_mask'][0],
            'labels': labels,
            'pixel_values': full_enc['pixel_values'],
            'image_grid_thw': full_enc['image_grid_thw'],
            'mm_token_type_ids': full_enc['mm_token_type_ids'][0],
        }


def collate_fn(batch):
    max_len = max(item['input_ids'].shape[0] for item in batch)
    pad_id = processor.tokenizer.pad_token_id

    input_ids, attn_mask, labels, mm_token_type_ids = [], [], [], []
    for item in batch:
        seq_len = item['input_ids'].shape[0]
        pad_len = max_len - seq_len
        input_ids.append(torch.cat([item['input_ids'], torch.full((pad_len,), pad_id, dtype=torch.long)]))
        attn_mask.append(torch.cat([item['attention_mask'], torch.zeros(pad_len, dtype=torch.long)]))
        labels.append(torch.cat([item['labels'], torch.full((pad_len,), -100, dtype=torch.long)]))
        mm_token_type_ids.append(torch.cat([item['mm_token_type_ids'], torch.zeros(pad_len, dtype=torch.long)]))

    pixel_values = torch.cat([item['pixel_values'] for item in batch], dim=0)
    image_grid_thw = torch.cat([item['image_grid_thw'] for item in batch], dim=0)

    return {
        'input_ids': torch.stack(input_ids),
        'attention_mask': torch.stack(attn_mask),
        'labels': torch.stack(labels),
        'pixel_values': pixel_values,
        'image_grid_thw': image_grid_thw,
        'mm_token_type_ids': torch.stack(mm_token_type_ids),
    }

In [15]:
import bitsandbytes as bnb

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.config.use_cache = False

FULL_MODEL_DIR = f'{OUTPUT_DIR}/qwen3vl8b_full_latest'

def safe_save_full_model(model, processor, path):
    tmp_path = path + '_tmp'
    if os.path.exists(tmp_path):
        shutil.rmtree(tmp_path)
    model.save_pretrained(
        tmp_path, safe_serialization=True, max_shard_size="4GB"
    )
    processor.save_pretrained(tmp_path)
    if os.path.exists(path):
        shutil.rmtree(path)
    os.rename(tmp_path, path)
    print(f"전체 모델 저장 완료: {path}")

tr_df = train[train['split'] == 'train'].reset_index(drop=True)
va_df = train[train['split'] == 'validation'].reset_index(drop=True)
print(f"Train: {len(tr_df)}, Val: {len(va_df)}")

tr_ds = QwenOrderingDataset(tr_df, f'{DATA_DIR}/train', processor)
va_ds = QwenOrderingDataset(va_df, f'{DATA_DIR}/train', processor)
TRAIN_NUM_WORKERS = min(8, max(2, (os.cpu_count() or 8) // 4))
tr_loader = DataLoader(
    tr_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=TRAIN_NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)
va_loader = DataLoader(
    va_ds,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=TRAIN_NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)
print(f"DataLoader workers={TRAIN_NUM_WORKERS}, prefetched_samples={TRAIN_NUM_WORKERS * 4}")

# 8-bit optimizer는 optimizer state만 줄이며 모델 파라미터는 전부 BF16으로 학습한다.
optimizer = bnb.optim.AdamW8bit(
    model.parameters(), lr=1e-5, betas=(0.9, 0.95), weight_decay=0.01
)
GRAD_ACCUM_STEPS = 16
MAX_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 1e-4
MAX_GRAD_NORM = 1.0

def evaluate_validation_loss(model, loader):
    model.eval()
    loss_sum = 0.0
    sample_count = 0
    with torch.inference_mode():
        for batch in tqdm(loader, desc="Validation loss", leave=False):
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            loss = model(**batch).loss
            if not torch.isfinite(loss):
                raise RuntimeError(f"비정상 validation loss: {loss.item()}")
            batch_size = batch['input_ids'].shape[0]
            loss_sum += loss.item() * batch_size
            sample_count += batch_size
    return loss_sum / sample_count

best_val_loss = float('inf')
best_epoch = 0
epochs_without_improvement = 0
training_history = []

for epoch in range(MAX_EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    for step, batch in enumerate(tqdm(tr_loader, desc=f"Epoch {epoch + 1}/{MAX_EPOCHS}")):
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        outputs = model(**batch)
        raw_loss = outputs.loss
        if not torch.isfinite(raw_loss):
            raise RuntimeError(f"비정상 loss 감지: step={step}, loss={raw_loss.item()}")
        (raw_loss / GRAD_ACCUM_STEPS).backward()

        is_update_step = ((step + 1) % GRAD_ACCUM_STEPS == 0) or (step + 1 == len(tr_loader))
        if is_update_step:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += raw_loss.item()
        if step % 100 == 0:
            peak_gb = torch.cuda.max_memory_allocated() / 1024**3
            print(f" step {step}: loss={raw_loss.item():.4f}, peak_alloc={peak_gb:.1f} GiB")

    train_loss = total_loss / len(tr_loader)
    val_loss = evaluate_validation_loss(model, va_loader)
    training_history.append({
        'epoch': epoch + 1, 'train_loss': train_loss, 'val_loss': val_loss
    })
    print(
        f"Epoch {epoch + 1}: train_loss={train_loss:.4f}, "
        f"val_loss={val_loss:.4f}"
    )

    if val_loss < best_val_loss - EARLY_STOPPING_MIN_DELTA:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        epochs_without_improvement = 0
        safe_save_full_model(model, processor, FULL_MODEL_DIR)
        print(f"새 best checkpoint: epoch={best_epoch}, val_loss={best_val_loss:.4f}")
    else:
        epochs_without_improvement += 1
        print(
            f"개선 없음: {epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch + 1}; best epoch={best_epoch}")
            break

print(f"Best epoch={best_epoch}, best validation loss={best_val_loss:.4f}")
print(f"최종 peak allocated: {torch.cuda.max_memory_allocated() / 1024**3:.1f} GiB")

Train: 7628, Val: 1907
DataLoader workers=8, prefetched_samples=32


Epoch 1/5:   0%|          | 0/7628 [00:00<?, ?it/s]

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


 step 0: loss=0.0007, peak_alloc=33.0 GiB
 step 100: loss=0.2244, peak_alloc=52.1 GiB
 step 200: loss=0.2308, peak_alloc=52.1 GiB
 step 300: loss=0.0312, peak_alloc=52.1 GiB
 step 400: loss=0.2352, peak_alloc=52.1 GiB
 step 500: loss=0.0905, peak_alloc=52.1 GiB
 step 600: loss=0.1365, peak_alloc=52.1 GiB
 step 700: loss=0.0382, peak_alloc=52.1 GiB
 step 800: loss=0.2967, peak_alloc=52.1 GiB
 step 900: loss=0.0674, peak_alloc=52.1 GiB
 step 1000: loss=0.3519, peak_alloc=52.1 GiB
 step 1100: loss=0.1203, peak_alloc=52.1 GiB
 step 1200: loss=0.0014, peak_alloc=52.1 GiB
 step 1300: loss=0.0261, peak_alloc=52.1 GiB
 step 1400: loss=0.2346, peak_alloc=52.1 GiB
 step 1500: loss=0.0141, peak_alloc=52.1 GiB
 step 1600: loss=0.0171, peak_alloc=52.1 GiB
 step 1700: loss=0.2050, peak_alloc=52.1 GiB
 step 1800: loss=0.0118, peak_alloc=52.1 GiB
 step 1900: loss=0.0895, peak_alloc=52.1 GiB
 step 2000: loss=0.0821, peak_alloc=52.1 GiB
 step 2100: loss=0.0708, peak_alloc=52.1 GiB
 step 2200: loss=0.112

Validation loss:   0%|          | 0/954 [00:00<?, ?it/s]

...

Epoch 2: train_loss=0.0943, val_loss=0.1119
개선 없음: 1/2


Epoch 3/5:   0%|          | 0/7628 [00:00<?, ?it/s]

 step 0: loss=0.0517, peak_alloc=52.1 GiB
 step 100: loss=0.0767, peak_alloc=52.1 GiB
 step 200: loss=0.0001, peak_alloc=52.1 GiB
 step 300: loss=0.1037, peak_alloc=52.1 GiB
 step 400: loss=0.0358, peak_alloc=52.1 GiB
 step 500: loss=0.0001, peak_alloc=52.1 GiB
 step 600: loss=0.0001, peak_alloc=52.1 GiB
 step 700: loss=0.0405, peak_alloc=52.1 GiB
 step 800: loss=0.0882, peak_alloc=52.1 GiB
 step 900: loss=0.0001, peak_alloc=52.1 GiB
 step 1000: loss=0.0054, peak_alloc=52.1 GiB
 step 1100: loss=0.0002, peak_alloc=52.1 GiB
 step 1200: loss=0.0000, peak_alloc=52.1 GiB
 step 1300: loss=0.2371, peak_alloc=52.1 GiB
 step 1400: loss=0.0000, peak_alloc=52.1 GiB
 step 1500: loss=0.0000, peak_alloc=52.1 GiB
 step 1600: loss=0.1939, peak_alloc=52.1 GiB
 step 1700: loss=0.0639, peak_alloc=52.1 GiB
 step 1800: loss=0.0000, peak_alloc=52.1 GiB
 step 1900: loss=0.0513, peak_alloc=52.1 GiB
 step 2000: loss=0.1436, peak_alloc=52.1 GiB
 step 2100: loss=0.1331, peak_alloc=52.1 GiB
 step 2200: loss=0.316

Validation loss:   0%|          | 0/954 [00:00<?, ?it/s]

Epoch 3: train_loss=0.0736, val_loss=0.1178
개선 없음: 2/2
Early stopping at epoch 3; best epoch=1
Best epoch=1, best validation loss=0.1114
최종 peak allocated: 52.1 GiB


In [16]:
import gc

# Release optimizer and training state before inference reload.
try:
    del optimizer
except NameError:
    pass
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

processor = AutoProcessor.from_pretrained(FULL_MODEL_DIR)
model = AutoModelForImageTextToText.from_pretrained(
    FULL_MODEL_DIR,
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
).to(DEVICE)
model.config.use_cache = True
model.eval()

assert not hasattr(model, 'peft_config')
assert not any('lora_' in name.lower() for name, _ in model.named_parameters())
print(f"Reloaded full fine-tuned model: {FULL_MODEL_DIR}")

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Reloaded full fine-tuned model: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/qwen3vl8b_full_latest


In [17]:
from scipy.stats import kendalltau

def compute_metrics(pred, target):
    exact_match = (pred == target).all(axis=1).mean()
    position_acc = (pred == target).mean()
    taus = [kendalltau(p,t)[0] if not np.isnan(kendalltau(p,t)[0]) else 1.0 for p,t in zip(pred,target)]
    return {'exact_match': exact_match, 'position_acc': position_acc, 'kendall_tau': np.mean(taus)}

processor.tokenizer.padding_side = 'left'  # 배치 생성을 위해 필수

BATCH_SIZE = 2

model.eval()
results = []

for start_idx in tqdm(range(0, len(va_df), BATCH_SIZE)):
    batch_rows = va_df.iloc[start_idx:start_idx + BATCH_SIZE]

    batch_messages = [get_prompt_message(row, f'{DATA_DIR}/train') for _, row in batch_rows.iterrows()]
    batch_texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in batch_messages]

    all_image_inputs = []
    for m in batch_messages:
        imgs, _ = process_vision_info(m, image_patch_size=IMAGE_PATCH_SIZE)
        all_image_inputs.extend(imgs)

    inputs = processor(
        text=batch_texts, images=all_image_inputs,
        padding=True, return_tensors="pt", do_resize=False
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)

    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    output_texts = processor.batch_decode(trimmed, skip_special_tokens=True)

    for (_, row), output_text in zip(batch_rows.iterrows(), output_texts):
        pred, parse_ok = parse_model_output(output_text)
        results.append({'Id': row['Id'], 'pred': pred, 'parse_ok': parse_ok})

pred_array = np.array([r['pred'] for r in results]) - 1
target_array = np.array([ast.literal_eval(a) for a in va_df['Answer']]) - 1

overall = compute_metrics(pred_array, target_array)
no_order_mask = va_df['No_ordering'].values
shuffled = compute_metrics(pred_array[~no_order_mask], target_array[~no_order_mask])

print(f"[Qwen3-VL-8B, MD5-group-stratified 80:20 val, n={len(va_df)}]")
print(f"  전체 exact={overall['exact_match']:.4f}")
print(f"  셔플됨 exact={shuffled['exact_match']:.4f}")
print(f"  파싱 실패율={1-np.mean([r['parse_ok'] for r in results]):.1%}")

  0%|          | 0/954 [00:00<?, ?it/s]

[Qwen3-VL-8B, MD5-group-stratified 80:20 val, n=1907]
  전체 exact=0.5244
  셔플됨 exact=0.4829
  파싱 실패율=0.0%


In [18]:
test = pd.read_csv(f'{DATA_DIR}/test.csv')
print(f"Test samples: {len(test)}")

Test samples: 819


In [19]:
import gc

torch.cuda.empty_cache()
gc.collect()

processor.tokenizer.padding_side = 'left'
BATCH_SIZE = 2
TEST_IMAGE_DIR = f'{DATA_DIR}/test'

# Keep completed rows when this cell is rerun.
if 'test_results' not in dir():
    test_results = []

processed_ids = set(r['Id'] for r in test_results)
remaining_test = test[~test['Id'].isin(processed_ids)].reset_index(drop=True)
print(f"Remaining: {len(remaining_test)} / {len(test)}")


def generate_batch(batch_rows):
    batch_messages = [
        get_prompt_message(row, TEST_IMAGE_DIR)
        for _, row in batch_rows.iterrows()
    ]
    batch_texts = [
        processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in batch_messages
    ]
    all_image_inputs = []
    for messages in batch_messages:
        images, _ = process_vision_info(
            messages, image_patch_size=IMAGE_PATCH_SIZE
        )
        all_image_inputs.extend(images)

    inputs = processor(
        text=batch_texts,
        images=all_image_inputs,
        padding=True,
        return_tensors="pt",
        do_resize=False,
    ).to(model.device)
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs, max_new_tokens=64, do_sample=False
        )
    trimmed = [
        output[len(input_ids):]
        for input_ids, output in zip(inputs.input_ids, generated_ids)
    ]
    return processor.batch_decode(trimmed, skip_special_tokens=True)


model.eval()
for start_idx in tqdm(range(0, len(remaining_test), BATCH_SIZE)):
    batch_rows = remaining_test.iloc[start_idx:start_idx + BATCH_SIZE]
    try:
        output_texts = generate_batch(batch_rows)
    except torch.cuda.OutOfMemoryError:
        # Rebuild every multimodal sample; slicing pixel_values would drop 3 of 4 images.
        torch.cuda.empty_cache()
        output_texts = []
        for row_idx in range(len(batch_rows)):
            one_row = batch_rows.iloc[row_idx:row_idx + 1]
            output_texts.extend(generate_batch(one_row))

    for (_, row), output_text in zip(batch_rows.iterrows(), output_texts):
        pred, parse_ok = parse_model_output(output_text)
        test_results.append({'Id': row['Id'], 'pred': pred, 'parse_ok': parse_ok})

print(f"Completed: {len(test_results)} / {len(test)}")
print(f"Parse failure rate: {1 - np.mean([r['parse_ok'] for r in test_results]):.1%}")

Remaining: 819 / 819


  0%|          | 0/410 [00:00<?, ?it/s]

Completed: 819 / 819
Parse failure rate: 0.0%


In [20]:
print(f"지금까지 처리된 개수: {len(test_results)}")

지금까지 처리된 개수: 819


In [21]:
predictions = [{'Id': r['Id'], 'Answer': str(r['pred'])} for r in test_results]
submission_df = pd.DataFrame(predictions)

submit_path = f'{OUTPUT_DIR}/submission_qwen3vl8b.csv'
submission_df.to_csv(submit_path, index=False)

# 형식 검증
sample_submission = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
assert list(submission_df.columns) == list(sample_submission.columns), "컬럼명 불일치!"
assert len(submission_df) == len(sample_submission), "행 개수 불일치!"
assert set(submission_df['Id']) == set(sample_submission['Id']), "Id 불일치!"

print(f"저장 완료: {submit_path}")
print("형식 검증 통과 — 제출 가능합니다.")
submission_df.head()

저장 완료: /content/drive/MyDrive/snu ai/snu_qwen3vl_outputs/submission_qwen3vl8b.csv
형식 검증 통과 — 제출 가능합니다.


,Id,Answer
0,WI8o3F,"[1, 4, 2, 3]"
1,Ae7zLm,"[2, 3, 4, 1]"
2,4QSIII,"[2, 4, 1, 3]"
3,k7h4gt,"[4, 1, 2, 3]"
4,ZtTwCx,"[1, 3, 2, 4]"


In [22]:
from collections import Counter

pred_strs = [str(r['pred']) for r in test_results]
print("전체 예측 분포 상위 5개:", Counter(pred_strs).most_common(5))

전체 예측 분포 상위 5개: [('[1, 2, 3, 4]', 128), ('[1, 3, 4, 2]', 43), ('[4, 1, 2, 3]', 38), ('[2, 1, 3, 4]', 38), ('[1, 2, 4, 3]', 37)]
